# Day 38: Web Search Grounding

Every answer so far came from the model's training data — frozen in time.

Today Gemini searches the live web, cites its sources, and answers with **current** information.

## Install

In [12]:
%pip install google-genai --quiet


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Setup

In [13]:
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='../.env')
client = genai.Client(api_key=os.environ["GEMINI_API_KEY2"])
print("\u2705 Client ready")

✅ Client ready


## Without Grounding

Ask about current events — the model can only guess from training data.

In [14]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="What were the top AI announcements this week? Give only 1 line points"
)
print(response.text)

Here are top AI announcements from this week in one line points:

*   **Anthropic** launched Claude 3.5 Sonnet, claiming it outperforms competitors like Claude 3 Opus and GPT-4o on key benchmarks.
*   The **EU AI Act** officially entered into force, setting a global precedent for comprehensive AI regulation.
*   **Perplexity AI** raised $67 million in new funding and launched an enterprise-grade AI search product.
*   **Character.AI** secured $100 million in new funding for its popular conversational AI platform.
*   **Apple's** "Apple Intelligence" announcements from WWDC continued to dominate discussions regarding on-device AI integration.


## Enable Google Search

Add `google_search` as a tool. Gemini now decides when to search the live web.

In [15]:
search_tool = types.Tool(google_search=types.GoogleSearch())

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="What were the top AI announcements this week? Give only 1 line points",
    config=types.GenerateContentConfig(tools=[search_tool])
)
print(response.text)

Here are the top AI announcements this week:

*   The U.S. Department of Commerce ordered a leading AI developer to suspend foreign national access to two of its frontier AI models (Fable 5 and Mythos 5) due to national security concerns regarding potential exploitability of safety guardrails.
*   A jailbreaker known as "Pliny the Liberator" reportedly bypassed Fable 5's safety classifiers using a "pack hunt" multi-agent attack, leading to the leak of its 120,000-character system prompt on GitHub.
*   Apple announced a major suite of AI features called "Apple Intelligence" at WWDC 2026, significantly enhancing Siri, photo editing, and image generation, with Siri now utilizing Google's Gemini model.
*   Google updated its research support tool, NotebookLM, to Gemini 3.5, adding new features for building source repositories from chats.
*   Anthropic published a paper titled "When AI Builds Itself," proposing a globally coordinated pause or slowdown on frontier AI development due to conce

In [16]:
print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""Here are the top AI announcements this week:

*   The U.S. Department of Commerce ordered a leading AI developer to suspend foreign national access to two of its frontier AI models (Fable 5 and Mythos 5) due to national security concerns regarding potential exploitability of safety guardrails.
*   A jailbreaker known as "Pliny the Liberator" reportedly bypassed Fable 5's safety classifiers using a "pack hunt" multi-agent attack, leading to the leak of its 120,000-character system prompt on GitHub.
*   Apple announced a major suite of AI features called "Apple Intelligence" at WWDC 2026, significantly enhancing Siri, photo editing, and image generation, with Siri now utilizing Google's Gemini model.
*   Google updated its research support tool, NotebookLM, to Gemini 3.5, adding new features for building source repositories from chats.
*   Anthropic 

## Test: Factual with Sources

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="What is the current population of India?",
    config=types.GenerateContentConfig(tools=[search_tool])
)
print(response.text)

## Test: Gemini Decides

For general knowledge, Gemini may skip searching. It only searches when it needs fresh data.

In [17]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="What is photosynthesis?",
    config=types.GenerateContentConfig(tools=[search_tool])
)
print(response.text)

Photosynthesis is a vital biochemical process used by plants, algae, and certain bacteria to convert light energy, typically from the sun, into chemical energy. This chemical energy is stored in organic compounds like sugars, which fuel the organisms' metabolism and growth.

Key components and the process:
*   **Ingredients:** Photosynthesis requires three main ingredients: carbon dioxide from the atmosphere, water absorbed from the environment, and energy from sunlight.
*   **Transformation:** During photosynthesis, these organisms capture light energy and use it to transform carbon dioxide and water into organic compounds such as glucose (a type of sugar). This process involves a series of light-dependent and light-independent reactions (like the Calvin cycle).
*   **Products:** The primary products of photosynthesis are glucose (which the organism uses for energy and building complex carbohydrates, lipids, and proteins) and oxygen, which is released as a byproduct into the atmospher

## Access Grounding Metadata

The response includes the sources Gemini cited.

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Who won the 2026 Champions League final?",
    config=types.GenerateContentConfig(tools=[search_tool])
)
print("Answer:", response.text)

if response.candidates[0].grounding_metadata:
    meta = response.candidates[0].grounding_metadata
    print("\n--- Sources ---")
    if meta.grounding_chunks:
        for chunk in meta.grounding_chunks:
            if chunk.web:
                print(f"  {chunk.web.title}: {chunk.web.uri}")

## Helper Function

Wrap it for reuse — ask anything with grounding.

In [ ]:
def ask_with_search(question: str) -> str:
    """Ask Gemini with Google Search grounding."""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=question,
        config=types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())]
        )
    )
    return response.text

print(ask_with_search("What is the latest Python version released?"))

## Key Takeaways

1. **`types.Tool(google_search=types.GoogleSearch())`** — one tool gives Gemini live web access
2. **Gemini decides when to search** — current events trigger search, general knowledge doesn't
3. **`grounding_metadata`** gives you the sources — titles, URLs, and which claims they support